# Control Flow - Match, Case

`match` compares one value against several **patterns**. It is **structural pattern matching**, available since Python 3.10.

| Pattern | Example | Matches |
|---|---|---|
| Literal | `case 404:` | That exact value |
| Wildcard | `case _:` | Anything (the default case) |
| Capture | `case x:` | Anything, and binds it to `x` |
| OR | `case 1 \| 2 \| 3:` | Any of the options |
| Sequence | `case [x, y]:` | A list or tuple of two items |
| Star | `case [first, *rest]:` | One or more items |
| Mapping | `case {"type": "click"}:` | A dict that has that key and value |
| Class | `case Point(x=0, y=0):` | An instance of the class with those attributes |
| Guard | `case n if n > 0:` | Matches only if the condition is true |
| AS | `case [1, 2] as pair:` | Matches and also binds the whole value |
| Dotted name | `case Color.RED:` | A constant value (not a capture) |

---

## Syntax

```python
match subject:
    case pattern1:
        # code
    case pattern2:
        # code
    case _:
        # default
```

### Important

* The subject is evaluated **once**.
* Cases are checked from top to bottom. The **first match wins**.
* There is **no fall-through**. Only one case block runs.
* If nothing matches and there is no `_`, nothing happens.
* `match` and `case` are **soft keywords**. They can still be used as normal variable names.

---

## Literal and OR Patterns

```python
match status:
    case 200:
        print("OK")
    case 301 | 302:
        print("Redirect")
    case 404:
        print("Not found")
    case _:
        print("Other")
```

---

## Capture Patterns

A bare name is a **capture**. It always matches and binds the value.

```python
match command:
    case "quit":
        ...
    case other:
        print(f"Unknown: {other}")
```

### Important

A bare name never compares against a variable. To match a constant, use a **dotted name** such as `Color.RED` or a literal.

---

## Sequence Patterns

```python
match point:
    case [0, 0]:
        print("Origin")
    case [x, 0]:
        print(f"On the x axis at {x}")
    case [x, y]:
        print(f"Point {x}, {y}")
    case [first, *rest]:
        print(first, rest)
```

* Works with lists and tuples.
* **Strings are not treated as sequences** in patterns.
* The length must match unless a `*` pattern is used.

---

## Mapping Patterns

```python
match event:
    case {"type": "click", "x": x, "y": y}:
        print(f"Click at {x}, {y}")
    case {"type": "key", "key": key}:
        print(f"Key {key}")
```

* Only the listed keys must exist. **Extra keys are ignored**.
* `**rest` collects the remaining items.

---

## Class Patterns

```python
match shape:
    case Circle(radius=r):
        ...
    case Rectangle(width=w, height=h):
        ...
```

* Keyword form (`Point(x=0)`) reads attributes by name.
* Positional form (`Point(0, 0)`) needs `__match_args__`. Dataclasses create it automatically.

---

## Guards

Add `if` after a pattern for an extra condition:

```python
match number:
    case n if n < 0:
        print("negative")
    case 0:
        print("zero")
    case n:
        print("positive")
```

---

## `if / elif` or `match`?

| Use `if / elif` for | Use `match` for |
|---|---|
| Simple true/false conditions | Comparing one value against many shapes |
| Ranges and complex boolean logic | Unpacking lists, dicts and objects |
| Two or three branches | Command parsers, message handlers, AST-like data |

## Source

https://docs.python.org/3/tutorial/controlflow.html#match-statements

https://docs.python.org/3/reference/compound_stmts.html#the-match-statement

In [ ]:
from dataclasses import dataclass
from enum import Enum

# Literal and OR patterns
def http_status(status):
    match status:
        case 200:
            return "OK"
        case 301 | 302:
            return "Redirect"
        case 404:
            return "Not found"
        case _:
            return "Other"

print([http_status(code) for code in (200, 302, 404, 500)])

# Capture, wildcard and guards
def describe(number):
    match number:
        case 0:
            return "zero"
        case n if n < 0:
            return f"negative {n}"
        case n:
            return f"positive {n}"

print(describe(0), describe(-4), describe(7))

# Sequence patterns
def where(point):
    match point:
        case [0, 0]:
            return "origin"
        case [x, 0]:
            return f"x axis at {x}"
        case [0, y]:
            return f"y axis at {y}"
        case [x, y]:
            return f"point {x}, {y}"
        case _:
            return "not a 2D point"

print(where((0, 0)), where([3, 0]), where((2, 5)), where([1, 2, 3]))

# Star patterns
def first_rest(items):
    match items:
        case []:
            return "empty"
        case [only]:
            return f"one item: {only}"
        case [first, *rest]:
            return f"first={first}, rest={rest}"

print(first_rest([]), first_rest([1]), first_rest([1, 2, 3]))

# Strings are not sequences in patterns
match "ab":
    case [a, b]:
        print("matched as a sequence")
    case str() as text:
        print("matched as a string:", text)

# Mapping patterns: extra keys are ignored
def handle(event):
    match event:
        case {"type": "click", "x": x, "y": y}:
            return f"click at {x}, {y}"
        case {"type": "key", "key": key}:
            return f"key {key}"
        case _:
            return "unknown event"

print(handle({"type": "click", "x": 1, "y": 2, "extra": True}), handle({"type": "key", "key": "a"}), handle({}))

# Class patterns (dataclasses provide __match_args__)
@dataclass
class Circle:
    radius: float

@dataclass
class Rectangle:
    width: float
    height: float

def area(shape):
    match shape:
        case Circle(radius=r):
            return round(3.14159 * r * r, 2)
        case Rectangle(w, h):
            return w * h
        case _:
            raise TypeError("unknown shape")

print(area(Circle(2)), area(Rectangle(3, 4)))

# Dotted names match constants; a bare name would capture instead
class Color(Enum):
    RED = 1
    GREEN = 2

def color_name(color):
    match color:
        case Color.RED:
            return "red"
        case Color.GREEN:
            return "green"

print(color_name(Color.RED), color_name(Color.GREEN))

# AS pattern and a command parser
def run(command):
    match command.split():
        case ["quit"]:
            return "bye"
        case ["go", ("north" | "south") as direction]:
            return f"going {direction}"
        case ["say", *words]:
            return " ".join(words)
        case _:
            return "unknown command"

print(run("quit"), run("go north"), run("say hello world"), run("go up"))

# match and case are soft keywords: still usable as names
match = "still a normal variable"
print(match)